In [0]:
%python
# Databricks notebook source
# COMMAND ----------
# DBTITLE 1,1. Parametrización del Entorno
dbutils.widgets.text("environment", "dev", "Ambiente de ejecución")
env = dbutils.widgets.get("environment")

print(f"--> Ejecutando pipeline Medallion en ambiente para prueba: {env.upper()}")

bronze_db = f"{env}_bronze"
silver_db = f"{env}_silver"
gold_db   = f"{env}_gold"

# COMMAND ----------
# DBTITLE 1,2. Definición Idempotente de Esquemas Medallion.
spark.sql(f"CREATE DATABASE IF NOT EXISTS {bronze_db}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {silver_db}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {gold_db}")

# COMMAND ----------
# DBTITLE 1,3. Capa Bronze (Ingesta Cruda)
from pyspark.sql.functions import current_timestamp

raw_data = [
    (101, "CAMPAIGN_A", 1200.0),
    (102, "CAMPAIGN_B", 3500.0),
    (102, "CAMPAIGN_B", 3500.0),  # Duplicado controlado
    (103, None, 450.0)             # Registro inválido
]
columns = ["campaign_id", "campaign_name", "budget"]

df_raw = spark.createDataFrame(raw_data, columns) \
              .withColumn("ingestion_timestamp", current_timestamp())

df_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{bronze_db}.raw_campaigns")

print(f"[OK] Capa Bronze generada en: {bronze_db}.raw_campaigns")

# COMMAND ----------
# DBTITLE 1,4. Capa Silver (Limpieza, Calidad y Tipado)
from pyspark.sql.functions import col, trim, upper

df_bronze = spark.read.table(f"{bronze_db}.raw_campaigns")

# Reglas de negocio Silver:
# 1. Filtro de integridad: descartar nulos o vacíos en nombre de campaña
# 2. Estandarización de texto (mayúsculas y sin espacios residuales)
# 3. Deduplicación por campaign_id
df_silver = (
    df_bronze
    .filter(col("campaign_name").isNotNull() & (trim(col("campaign_name")) != ""))
    .withColumn("campaign_name", upper(trim(col("campaign_name"))))
    .dropDuplicates(["campaign_id"])
    .select(
        col("campaign_id").cast("integer"),
        col("campaign_name").cast("string"),
        col("budget").cast("double"),
        col("ingestion_timestamp")
    )
)

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_db}.clean_campaigns")

print(f"[OK] Capa Silver generada en: {silver_db}.clean_campaigns")

# COMMAND ----------
# DBTITLE 1,5. Capa Gold (Agregación Analítica / Consumo BI)
from pyspark.sql.functions import count, sum as _sum, round as _round

df_silver_read = spark.read.table(f"{silver_db}.clean_campaigns")

# Reglas Gold:
# Agregación analítica por campaña lista para dashboards
df_gold = (
    df_silver_read
    .groupBy("campaign_name")
    .agg(
        count("campaign_id").alias("total_campaign_instances"),
        _round(_sum("budget"), 2).alias("total_allocated_budget")
    )
    .withColumn("processed_at", current_timestamp())
)

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_db}.campaign_summary")

print(f"[OK] Capa Gold generada en: {gold_db}.campaign_summary")